# California Housing Dataset

Source: `sklearn.datasets.fetch_california_housing`  
20,640 samples, no missing values.

## Raw Features

| Feature | Description |
|---|---|
| MedInc | Median income in the block group (tens of thousands USD) |
| HouseAge | Median house age in the block group (years) |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Block group population |
| AveOccup | Average household occupancy (persons per household) |
| Latitude | Block group latitude |
| Longitude | Block group longitude |

## Target

| Column | Description |
|---|---|
| MedHouseVal | Median house value in units of $100,000 |

## Datasets Produced

**Dataset 1 - Linear Regression** (`housing_lr.csv`)  
Uses the 8 raw features to predict `MedHouseVal` as a continuous value. Split 80/20 into train and test sets. Features are standard-scaled before export.

**Dataset 2 - KNN Classification** (`housing_knn.csv`)  
Uses the 8 raw features plus 4 engineered features to classify each house into one of 5 categories. The target column is a class label (0-4) derived from quantile-based thresholds. `MedHouseVal` is used only for label assignment and excluded from the feature set to avoid leakage. Split 80/20 into train and test sets. Features are standard-scaled before export.

In [2]:
%pip install pandas numpy scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 14.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 27.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [7]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [8]:
housing = fetch_california_housing(as_frame=True)

features_df = housing.data.copy()
target_df = housing.target.rename("MedHouseVal")

print(features_df.shape, target_df.shape)
features_df.head()

(20640, 8) (20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


# Dataset 1 - Linear Regression

The raw 8 features are standard-scaled (zero mean, unit variance) to ensure gradient descent converges uniformly across features with different scales (e.g. `Population` in the hundreds vs `MedInc` in the single digits).

The target `MedHouseVal` is left unscaled so predictions remain interpretable in units of $100,000.

The dataset is split 80/20. Both train and test CSVs have no header, with `MedHouseVal` as the last column.

In [ ]:
scaler_lr = StandardScaler()
features_scaled_lr = scaler_lr.fit_transform(features_df)

df_lr = pd.DataFrame(features_scaled_lr, columns=features_df.columns)
df_lr["MedHouseVal"] = target_df.values

df_lr_train, df_lr_test = train_test_split(df_lr, test_size=0.2, random_state=42)

df_lr_train.to_csv("../data/housing_lr_train.csv", header=False, index=False)
df_lr_test.to_csv("../data/housing_lr_test.csv", header=False, index=False)

print("train:", df_lr_train.shape, "test:", df_lr_test.shape)

train: (16512, 9) test: (4128, 9)


# Dataset 2 - KNN Classification

## Engineered Features

Four features are derived from the raw columns to capture housing density and affordability structure:

| Feature | Formula | Description |
|---|---|---|
| occupancy_density | AveOccup | Persons per household (alias for clarity) |
| rooms_per_capita | AveRooms / AveOccup | Total rooms divided by population, household count cancels |
| bedroom_ratio | AveBedrms / AveRooms | Fraction of rooms that are bedrooms, household count cancels |
| dist_to_coast | haversine to nearest coastal city | Minimum distance in km to a set of reference coastal points |

`cost_to_income` (MedHouseVal / MedInc) is computed for label assignment only and excluded from the feature set to avoid target leakage.

## Class Labels

Labels are assigned by quantile-based thresholds applied in priority order. The first matching rule wins:

| Priority | Label | Class | Conditions |
|---|---|---|---|
| 1 | 0 | Luxury | cost_to_income >= p66, MedInc >= p66, bedroom_ratio <= p33, dist_to_coast <= p25 |
| 2 | 4 | Unaffordable | cost_to_income >= p66, MedInc < p66 (high cost relative to local income) |
| 3 | 3 | Working Class | MedHouseVal <= p33, occupancy_density >= p66, bedroom_ratio >= p66 |
| 4 | 1 | Comfortable | MedHouseVal >= p66, MedInc >= p66, occupancy_density <= p33 |
| 5 | 2 | Cost-effective | default (does not meet any of the above conditions) |

The priority order matters: Luxury is a strict subset of high cost-to-income houses, so it is checked before Unaffordable to prevent misclassification.

## Features Used

The 12 features written to CSV are the 8 raw features plus the 4 engineered features above. `MedHouseVal` and `cost_to_income` are excluded. All 12 features are standard-scaled before export.

In [9]:
COASTAL_POINTS = [
    (32.7157, -117.1611),  # San Diego
    (34.0522, -118.2437),  # Los Angeles
    (34.4208, -119.6982),  # Santa Barbara
    (35.3658, -120.8499),  # Morro Bay
    (36.6002, -121.8947),  # Monterey
    (36.9741, -122.0308),  # Santa Cruz
    (37.7749, -122.4194),  # San Francisco
    (40.8021, -124.1637),  # Eureka
]

def haversine_km(lat1, lon1, lat2, lon2):
    """
    Computes the great-circle distance in km between two lat/lon points.
    Input: four floats in decimal degrees
    Output: distance in km
    """
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def dist_to_coast(lat, lon):
    """
    Returns the minimum haversine distance in km from a point to any coastal reference.
    Input: lat, lon in decimal degrees
    Output: minimum distance in km
    """
    dists = [haversine_km(lat, lon, plat, plon) for plat, plon in COASTAL_POINTS]
    return min(dists)

features_df["dist_to_coast"] = features_df.apply(
    lambda r: dist_to_coast(r["Latitude"], r["Longitude"]), axis=1
)

In [10]:
df_knn = features_df.copy()
df_knn["MedHouseVal"] = target_df.values

df_knn["occupancy_density"]  = df_knn["AveOccup"]
df_knn["rooms_per_capita"]   = df_knn["AveRooms"] / df_knn["AveOccup"]
df_knn["bedroom_ratio"]      = df_knn["AveBedrms"] / df_knn["AveRooms"]
df_knn["cost_to_income"]     = df_knn["MedHouseVal"] / df_knn["MedInc"]

df_knn.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,dist_to_coast,MedHouseVal,occupancy_density,rooms_per_capita,bedroom_ratio,cost_to_income
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,20.329542,4.526,2.555556,2.732919,0.146591,0.543651
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,19.908065,3.585,2.109842,2.956685,0.155797,0.431855
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,17.835390,3.521,2.802260,2.957661,0.129516,0.485160
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,17.064123,3.413,2.547945,2.283154,0.184458,0.604809
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,17.064123,3.422,2.181467,2.879646,0.172096,0.889709


In [11]:
q = lambda col, p: df_knn[col].quantile(p)

thr = {
    "cost_to_income_hi":  q("cost_to_income",    0.66),
    "medinc_hi":          q("MedInc",            0.66),
    "bedroom_ratio_lo":   q("bedroom_ratio",     0.33),
    "bedroom_ratio_hi":   q("bedroom_ratio",     0.66),
    "density_hi":         q("occupancy_density", 0.66),
    "density_lo":         q("occupancy_density", 0.33),
    "value_lo":           q("MedHouseVal",       0.33),
    "value_hi":           q("MedHouseVal",       0.66),
    "coast_near":         q("dist_to_coast",     0.25),
}

thr

{'cost_to_income_hi': np.float64(0.5862495743950349),
 'medinc_hi': np.float64(4.214744),
 'bedroom_ratio_lo': np.float64(0.18462522986392058),
 'bedroom_ratio_hi': np.float64(0.22461147749615212),
 'density_hi': np.float64(3.0850636654891974),
 'density_lo': np.float64(2.561454216937693),
 'value_lo': np.float64(1.406),
 'value_hi': np.float64(2.28),
 'coast_near': np.float64(17.887392148046672)}

In [12]:
def assign_label(row):
    """
    Assigns a housing class label based on quantile thresholds.
    Rules are evaluated in priority order — first match wins.
    Input: a DataFrame row with engineered features and MedHouseVal
    Output: integer label 0-4
    """
    is_coastal = row["dist_to_coast"] <= thr["coast_near"]

    if (row["cost_to_income"]  >= thr["cost_to_income_hi"] and
        row["MedInc"]          >= thr["medinc_hi"]          and
        row["bedroom_ratio"]   <= thr["bedroom_ratio_lo"]   and
        is_coastal):
        return 0  # Luxury

    if (row["cost_to_income"] >= thr["cost_to_income_hi"] and
        row["MedInc"]         <  thr["medinc_hi"]):
        return 4  # Unaffordable

    if (row["MedHouseVal"]        <= thr["value_lo"]      and
        row["occupancy_density"]  >= thr["density_hi"]    and
        row["bedroom_ratio"]      >= thr["bedroom_ratio_hi"]):
        return 3  # Working Class

    if (row["MedHouseVal"]       >= thr["value_hi"]   and
        row["MedInc"]            >= thr["medinc_hi"]   and
        row["occupancy_density"] <= thr["density_lo"]):
        return 1  # Comfortable

    return 2  # Cost-effective (default)

df_knn["label"] = df_knn.apply(assign_label, axis=1)
df_knn["label"].value_counts().sort_index()

label
0      321
1     1420
2    12593
3     1037
4     5269
Name: count, dtype: int64

In [14]:
knn_feature_cols = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude",
    "occupancy_density", "rooms_per_capita", "bedroom_ratio", "dist_to_coast",
]

df_knn_final = df_knn[knn_feature_cols + ["label"]]

scaler_knn = StandardScaler()
features_scaled_knn = scaler_knn.fit_transform(df_knn_final[knn_feature_cols])

df_knn_scaled = pd.DataFrame(features_scaled_knn, columns=knn_feature_cols)
df_knn_scaled["label"] = df_knn_final["label"].values

df_knn_train, df_knn_test = train_test_split(df_knn_scaled, test_size=0.2, random_state=42)

df_knn_train.to_csv("../data/housing_knn_train.csv", header=False, index=False)
df_knn_test.to_csv("../data/housing_knn_test.csv", header=False, index=False)

print("train:", df_knn_train.shape, "test:", df_knn_test.shape)

train: (16512, 13) test: (4128, 13)
